# 03. Training (Text Cell)

ADR-016/017 §3-2 model × polluter × level × dataset 학습 → metric csv.

분류 5종: LogReg+TFIDF / TextCNN / DistilBERT / BERT / RoBERTa
회귀 5종: Ridge+TFIDF / XGBoost+TFIDF / TextCNN-Reg / DistilBERT-Reg / BERT-Reg

학습 함수는 `dsc_framework.text_trainers`에서 import (검증 완료).

GPU 시간: 분류 30~50시간 + 회귀 30~50시간 = 60~100시간 (T4 가정).
Colab Pro+ 또는 Pay-as-you-go.

Output: `results/text_train_metrics.csv`


In [ ]:
# ============================================================
# 0. Drive 마운트 + dsc/ 자동 검색 + sys.path 등록
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, glob, json
import numpy as np
import pandas as pd


def _find_dsc_base():
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None
    for c in [f'{root}/capstone/dsc', f'{root}/dsc', f'{root}/capstone-dsc']:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pat in [f'{root}/*/dsc_framework/__init__.py',
                f'{root}/*/*/dsc_framework/__init__.py',
                f'{root}/*/*/*/dsc_framework/__init__.py']:
        for hit in glob.glob(pat):
            return os.path.dirname(os.path.dirname(hit))
    return None


BASE = _find_dsc_base()
if BASE is None:
    drive_root = '/content/drive/MyDrive'
    listing = os.listdir(drive_root) if os.path.isdir(drive_root) else []
    raise RuntimeError(
        'dsc_framework/ 폴더를 G드라이브에서 못 찾음.\n'
        '  1) G드라이브 클라이언트 sync 완료 확인 (commit 직후면 잠시 대기 후 재시도)\n'
        '  2) Drive 마운트 확인 — !ls /content/drive/MyDrive\n'
        f'  현재 Drive 내용: {listing[:20]}'
    )

# 누락 파일 진단 — partial sync 시 빠른 실패
REQUIRED = ['shared_metrics.py', 'classification_cell.py', 'regression_cell.py',
            'image_cell.py', 'text_cell.py', 'text_cell_regression.py',
            'text_trainers.py', 'data_type_detection.py', 'router.py',
            'text_polluters', 'image_polluters']
missing = [f for f in REQUIRED if not os.path.exists(f'{BASE}/dsc_framework/{f}')]
if missing:
    raise RuntimeError(
        f'dsc_framework/ 파일 누락: {missing}\n'
        '→ G드라이브 sync 미완료. 잠시 대기 후 재실행.\n'
        '→ Colab Drive view stale 시: drive.flush_and_unmount() 후 재마운트.'
    )

RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/text'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

print(f'BASE: {BASE}')
print(f'dsc_framework 파일: {sorted(f for f in os.listdir(f"{BASE}/dsc_framework") if not f.startswith("_"))}')


In [ ]:
# ============================================================
# 의존성 설치 (Colab 1회 실행 후 다음 셀)
# ============================================================
%pip install -q 'transformers>=4.30' 'datasets>=2.10' 'xgboost>=1.7' 'accelerate>=1.1.0'


In [ ]:
# ============================================================
# imports
# ============================================================
from dsc_framework.text_trainers import (
    CLASSIFICATION_MODELS, REGRESSION_MODELS,
)
from dsc_framework.text_polluters import (
    CompletenessTextPolluter, NoiseInjectionTextPolluter, WordShufflePolluter,
    ClassBalanceTextPolluter, LabelSwapTextPolluter,
    TargetDistributionSkewTextPolluter, TargetNoiseTextPolluter,
)
print('trainers OK:', list(CLASSIFICATION_MODELS.keys()), list(REGRESSION_MODELS.keys()))


In [ ]:
# ============================================================
# 데이터셋 로드 (ADR-016 분류 3종 + ADR-017 회귀 3종)
#   - Phase 2 정식 실행 시 N_TRAIN/N_TEST를 ADR §4 sample_cap으로 키울 것
#   - sanity/dev에선 작은 sample로 시작
# ============================================================
from datasets import load_dataset
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

# 사용자가 sample size 조정. ADR §4 정식 = train 50000~200000 / test 5000~50000
N_TRAIN = 3000  # 분류·SST5는 자동 cap, 빈 dataset이면 자체 train size
N_TEST  = 500


def _slice(ds_dict, split, n, seed=42):
    ds = ds_dict[split] if split in ds_dict else ds_dict['train']
    if n is None or len(ds) <= n:
        return ds
    return ds.shuffle(seed=seed).select(range(n))


def load_all():
    """분류 3 + 회귀 3 = 6 dataset을 (tr_texts, tr_y, te_texts, te_y, task)로 반환."""
    out = {}

    # 분류
    ag = load_dataset('fancyzhx/ag_news')
    out['ag_news'] = (_slice(ag, 'train', N_TRAIN), _slice(ag, 'test', N_TEST), 'classification')

    imdb = load_dataset('stanfordnlp/imdb')
    out['imdb'] = (_slice(imdb, 'train', N_TRAIN), _slice(imdb, 'test', N_TEST), 'classification')

    news20 = load_dataset('SetFit/20_newsgroups')
    out['20news'] = (_slice(news20, 'train', N_TRAIN), _slice(news20, 'test', N_TEST), 'classification')

    # 회귀 (label = star/sentiment를 float)
    yelp = load_dataset('Yelp/yelp_review_full')
    out['yelp_full'] = (_slice(yelp, 'train', N_TRAIN), _slice(yelp, 'test', N_TEST), 'regression')

    amazon = load_dataset('SetFit/amazon_reviews_multi_en')  # ADR-017 미러
    out['amazon_en'] = (_slice(amazon, 'train', N_TRAIN), _slice(amazon, 'test', N_TEST), 'regression')

    sst = load_dataset('SetFit/sst5')
    out['sst5'] = (_slice(sst, 'train', N_TRAIN), _slice(sst, 'test', N_TEST), 'regression')

    return out


def to_lists(ds_split, task):
    texts = ds_split['text']
    labels = ds_split['label']
    if task == 'regression':
        labels = [float(y) for y in labels]
    return list(texts), list(labels)


print('load_dataset OK — load_all() 호출하면 6 dataset 로드 시작.')


In [ ]:
# ============================================================
# 학습 sweep 설정
# ============================================================
LEVEL_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9]
SEED = 42  # 학습은 1 seed (NB_02 sweep과 메모리 절약)

POLLUTERS_CLS = {
    'completeness_text':    CompletenessTextPolluter,
    'noise_injection_text': NoiseInjectionTextPolluter,
    'word_shuffle':         WordShufflePolluter,
    'class_balance':        ClassBalanceTextPolluter,
    'label_swap':           LabelSwapTextPolluter,
}
POLLUTERS_REG = {
    'completeness_text':         CompletenessTextPolluter,
    'noise_injection_text':      NoiseInjectionTextPolluter,
    'word_shuffle':              WordShufflePolluter,
    'target_distribution_skew':  TargetDistributionSkewTextPolluter,
    'target_noise':              TargetNoiseTextPolluter,
}


def train_one_combo(model_name, train_fn, tr_t, tr_y, te_t, te_y, task, **kw):
    try:
        metric = train_fn(tr_t, tr_y, te_t, te_y, **kw)
        return {'metric': float(metric), 'error': None}
    except Exception as e:
        return {'metric': float('nan'), 'error': f'{type(e).__name__}: {e}'}


def model_sweep(name, tr_ds, te_ds, task):
    tr_texts, tr_y = to_lists(tr_ds, task)
    te_texts, te_y = to_lists(te_ds, task)
    polluters = POLLUTERS_CLS if task == 'classification' else POLLUTERS_REG
    models = CLASSIFICATION_MODELS if task == 'classification' else REGRESSION_MODELS
    rows = []
    for pol_name, pol_cls in polluters.items():
        for lvl in LEVEL_GRID:
            pol = pol_cls(lvl, random_seed=SEED)
            tr_p, tr_yp = pol.pollute(tr_texts, tr_y)
            for model_name, train_fn in models.items():
                t0 = __import__('time').time()
                res = train_one_combo(model_name, train_fn, tr_p, tr_yp,
                                       te_texts, te_y, task)
                rows.append({
                    'dataset': name, 'task': task, 'model': model_name,
                    'polluter': pol_name, 'level': lvl, 'seed': SEED,
                    'metric': res['metric'], 'error': res['error'],
                    'elapsed_s': round(__import__('time').time() - t0, 2),
                })
                print(f'  {model_name:14s} pol={pol_name:24s} lvl={lvl:.2f} metric={res["metric"]:.3f}')
    return rows


In [ ]:
# ============================================================
# 전체 학습 sweep — 매우 long-running. 6 dataset × 5 model × 5 pol × 6 lvl
# = 900 학습. GPU 큐 백그라운드 권장. 체크포인트 csv 저장.
# ============================================================
import time
datasets = load_all()

ckpt = f'{RESULTS_DIR}/text_train_metrics.csv'
all_rows = []

for name, (tr_ds, te_ds, task) in datasets.items():
    print(f'\n=== {name} ({task}) 학습 시작 ===')
    t0 = time.time()
    rows = model_sweep(name, tr_ds, te_ds, task)
    all_rows.extend(rows)
    pd.DataFrame(all_rows).to_csv(ckpt, index=False)  # 체크포인트
    print(f'  {len(rows)} rows, {(time.time()-t0)/60:.1f}분')

print(f'\nfinal: {ckpt} ({len(all_rows)} rows)')
pd.DataFrame(all_rows).head()


---

다음: `04_scoreboard_text.ipynb` — r·hold-out·default vs tuned.
